## Phase 3 : SQL Data Analysis

#### Goal : Use SQL Queries to extract insights from a dataset.

Key Requirements : 
- Write SELECT queries 
- Use WHERE, ORDER BY, GROUP BY
- Perform basic aggregations(COUNT, SUM, AVG)


### Import and dataset setup

In [22]:
import pandas as pd
import sqlite3

df = pd.read_excel('Dataset_Cleaned.xlsx')

conn = sqlite3.connect(':memory:')

df.to_sql('orders', conn, index=False, if_exists='replace')

print('Database Ready!')
print(f'Total rows loaded : {len(df)}')

Database Ready!
Total rows loaded : 1200


### Helper function

In [23]:
def run_query(sql):
    result = pd.read_sql_query(sql, conn)
    return result

### SELECT Query

In [24]:
print("=== QUERY 1: First 10 orders ===")
run_query("""
    SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice, OrderStatus
    FROM orders
    LIMIT 10
""")

=== QUERY 1: First 10 orders ===


,OrderID,Product,Quantity,UnitPrice,TotalPrice,OrderStatus
0,ORD200000,Monitor,5,570.62,2853.10,Shipped
1,ORD200001,Phone,2,151.35,302.70,Shipped
2,ORD200002,Tablet,5,550.68,2753.40,Cancelled
3,ORD200003,Chair,1,273.19,273.19,Returned
4,ORD200004,Printer,4,626.01,2504.04,Delivered
5,ORD200005,Phone,2,245.86,491.72,Shipped
6,ORD200006,Laptop,1,664.42,664.42,Returned
7,ORD200007,Monitor,5,149.55,747.75,Shipped
8,ORD200008,Phone,2,134.28,268.56,Cancelled
9,ORD200009,Desk,4,509.38,2037.52,Shipped


### WHERE Clause

In [25]:
print("=== QUERY 2: Sirf Delivered orders ===")
run_query("""
    SELECT OrderID, Product, TotalPrice, OrderStatus
    FROM orders
    WHERE OrderStatus = 'Delivered'
    ORDER BY TotalPrice DESC
    LIMIT 10
""")

=== QUERY 2: Sirf Delivered orders ===


,OrderID,Product,TotalPrice,OrderStatus
0,ORD200789,Tablet,3456.40,Delivered
1,ORD200632,Laptop,3390.80,Delivered
2,ORD201065,Printer,3334.00,Delivered
3,ORD200361,Printer,3299.25,Delivered
4,ORD200837,Chair,3277.75,Delivered
5,ORD201079,Phone,3170.00,Delivered
6,ORD200511,Monitor,2876.20,Delivered
7,ORD200578,Monitor,2830.35,Delivered
8,ORD200883,Printer,2807.40,Delivered
9,ORD200769,Printer,2714.20,Delivered


In [26]:
print("=== QUERY 3: High value orders (TotalPrice > 2000) ===")
run_query("""
    SELECT OrderID, Product, Quantity, TotalPrice
    FROM orders
    WHERE TotalPrice > 2000
    ORDER BY TotalPrice DESC
""")

=== QUERY 3: High value orders (TotalPrice > 2000) ===


,OrderID,Product,Quantity,TotalPrice
0,ORD200789,Tablet,5,3456.40
1,ORD201122,Monitor,5,3390.95
2,ORD200632,Laptop,5,3390.80
3,ORD200469,Chair,5,3384.90
4,ORD200328,Tablet,5,3370.20
...,...,...,...,...
175,ORD201035,Laptop,4,2024.44
176,ORD200181,Printer,5,2021.95
177,ORD200915,Laptop,4,2011.96
178,ORD200979,Monitor,3,2007.18


In [27]:
print("=== QUERY 4: Orders from Instagram and Used the coupons too ===")
run_query("""
    SELECT OrderID, Product, TotalPrice, ReferralSource, CouponCode
    FROM orders
    WHERE ReferralSource = 'Instagram'
    AND CouponCode != 'No Coupon'
    LIMIT 10
""")

=== QUERY 4: Orders from Instagram and Used the coupons too ===


,OrderID,Product,TotalPrice,ReferralSource,CouponCode
0,ORD200000,Monitor,2853.10,Instagram,SAVE10
1,ORD200005,Phone,491.72,Instagram,SAVE10
2,ORD200010,Tablet,3129.85,Instagram,WINTER15
3,ORD200014,Tablet,786.66,Instagram,SAVE10
4,ORD200020,Chair,312.55,Instagram,WINTER15
5,ORD200037,Laptop,153.48,Instagram,SAVE10
6,ORD200040,Desk,688.04,Instagram,FREESHIP
7,ORD200041,Monitor,1834.35,Instagram,FREESHIP
8,ORD200046,Monitor,2200.56,Instagram,FREESHIP
9,ORD200050,Tablet,179.22,Instagram,FREESHIP


### GROUP BY and Aggregation

In [28]:
print("=== QUERY 5: Total revenue of each product ===")
run_query("""
    SELECT Product,
           COUNT(*) AS total_orders,
           SUM(TotalPrice) AS total_revenue,
           AVG(TotalPrice) AS avg_order_value
    FROM orders
    GROUP BY Product
    ORDER BY total_revenue DESC
""")

=== QUERY 5: Total revenue of each product ===


,Product,total_orders,total_revenue,avg_order_value
0,Chair,178,195620.11,1098.989382
1,Printer,181,195612.61,1080.732652
2,Laptop,173,192126.56,1110.558150
3,Tablet,179,186568.95,1042.284637
4,Monitor,163,175651.41,1077.616012
5,Desk,170,167459.93,985.058412
6,Phone,156,151722.39,972.579423


In [29]:
print("=== QUERY 6: Orders in each order status ===")
run_query("""
    SELECT OrderStatus,
           COUNT(*) AS order_count,
           SUM(TotalPrice) AS revenue_at_risk
    FROM orders
    GROUP BY OrderStatus
    ORDER BY order_count DESC
""")

=== QUERY 6: Orders in each order status ===


,OrderStatus,order_count,revenue_at_risk
0,Cancelled,250,276396.21
1,Returned,247,243277.70
2,Pending,237,256328.15
3,Shipped,235,246159.58
4,Delivered,231,242600.32


In [30]:
print("=== QUERY 7: Referral Source performance ===")
run_query("""
    SELECT ReferralSource,
           COUNT(*) AS total_orders,
           ROUND(SUM(TotalPrice), 2) AS total_revenue
    FROM orders
    GROUP BY ReferralSource
    ORDER BY total_orders DESC
""")

=== QUERY 7: Referral Source performance ===


,ReferralSource,total_orders,total_revenue
0,Instagram,259,275285.45
1,Email,250,261808.55
2,Google,241,250441.48
3,Facebook,228,250410.90
4,Referral,222,226815.58


### HAVING (filter AFTER grouping)

In [31]:
print("=== QUERY 8: Products having total revenue greater than 1,00,000 ===")
run_query("""
    SELECT Product,
           COUNT(*) AS total_orders,
           ROUND(SUM(TotalPrice), 2) AS total_revenue
    FROM orders
    GROUP BY Product
    HAVING total_revenue > 100000
    ORDER BY total_revenue DESC
""")

=== QUERY 8: Products having total revenue greater than 1,00,000 ===


,Product,total_orders,total_revenue
0,Chair,178,195620.11
1,Printer,181,195612.61
2,Laptop,173,192126.56
3,Tablet,179,186568.95
4,Monitor,163,175651.41
5,Desk,170,167459.93
6,Phone,156,151722.39


In [32]:
print("=== QUERY 9: Payment methods having orders more than 200 ===")
run_query("""
    SELECT PaymentMethod,
           COUNT(*) AS order_count
    FROM orders
    GROUP BY PaymentMethod
    HAVING order_count > 200
    ORDER BY order_count DESC
""")


=== QUERY 9: Payment methods having orders more than 200 ===


,PaymentMethod,order_count
0,Online,258
1,Cash,246
2,Credit Card,234
3,Debit Card,232
4,Gift Card,230


### Business Insight Queries (So What?)

In [33]:
print("=== QUERY 10: Coupon vs Non-coupon revenue comparison ===")
run_query("""
    SELECT 
        CASE WHEN CouponCode = 'No Coupon' THEN 'No Coupon' 
             ELSE 'Used Coupon' END AS coupon_used,
        COUNT(*) AS orders,
        ROUND(AVG(TotalPrice), 2) AS avg_order_value,
        ROUND(SUM(TotalPrice), 2) AS total_revenue
    FROM orders
    GROUP BY coupon_used
""")

=== QUERY 10: Coupon vs Non-coupon revenue comparison ===


,coupon_used,orders,avg_order_value,total_revenue
0,No Coupon,309,1043.37,322401.41
1,Used Coupon,891,1057.64,942360.55


In [34]:
print("=== QUERY 11: Monthly revenue trend ===")
run_query("""
    SELECT 
        SUBSTR(Date, 1, 7) AS month,
        COUNT(*) AS orders,
        ROUND(SUM(TotalPrice), 2) AS monthly_revenue
    FROM orders
    GROUP BY month
    ORDER BY month ASC
""")

=== QUERY 11: Monthly revenue trend ===


,month,orders,monthly_revenue
0,2023-01,47,56685.75
1,2023-02,37,40117.66
2,2023-03,43,48609.37
3,2023-04,31,27751.71
4,2023-05,49,63836.84
5,2023-06,45,49500.19
6,2023-07,44,42820.66
7,2023-08,51,54352.14
8,2023-09,29,29526.67
9,2023-10,47,52607.85


### FINAL SUMMARY

In [35]:
print("=" * 50)
print("PROJECT 3 — SQL ANALYSIS SUMMARY")
print("=" * 50)

total_revenue = pd.read_sql_query("SELECT SUM(TotalPrice) FROM orders", conn).iloc[0,0]
total_orders = pd.read_sql_query("SELECT COUNT(*) FROM orders", conn).iloc[0,0]
avg_order = pd.read_sql_query("SELECT AVG(TotalPrice) FROM orders", conn).iloc[0,0]
top_product = pd.read_sql_query("SELECT Product FROM orders GROUP BY Product ORDER BY SUM(TotalPrice) DESC LIMIT 1", conn).iloc[0,0]

print(f"\nTotal Orders    : {total_orders}")
print(f"Total Revenue   : ₹{total_revenue:,.2f}")
print(f"Avg Order Value : ₹{avg_order:.2f}")
print(f"Top Product     : {top_product}")

conn.close()
print("\nDatabase connection closed.")

PROJECT 3 — SQL ANALYSIS SUMMARY

Total Orders    : 1200
Total Revenue   : ₹1,264,761.96
Avg Order Value : ₹1053.97
Top Product     : Chair

Database connection closed.
